In [1]:
# ============================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================

import os
import glob
import numpy as np
import mne

# Reproducibility
np.random.seed(42)

# --------------------------------------------
# Dataset location
# --------------------------------------------

BASE_DIR = r"D:\EEG MIT\chb-mit-scalp-eeg-database-1.0.0\chb-mit-scalp-eeg-database-1.0.0"

# --------------------------------------------
# Patient split
# --------------------------------------------

TRAIN_PATIENTS = [
    f"chb{i:02d}" for i in range(1, 18)
]

TEST_PATIENTS = [
    f"chb{i:02d}" for i in range(18, 25)
]

print("Training patients:")
print(TRAIN_PATIENTS)

print("\nTesting patients:")
print(TEST_PATIENTS)

print("\nNumber of training patients:", len(TRAIN_PATIENTS))
print("Number of testing patients:", len(TEST_PATIENTS))

Training patients:
['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14', 'chb15', 'chb16', 'chb17']

Testing patients:
['chb18', 'chb19', 'chb20', 'chb21', 'chb22', 'chb23', 'chb24']

Number of training patients: 17
Number of testing patients: 7


In [2]:
# ============================================
# CELL 2 — PATIENT DIRECTORIES
# ============================================

ALL_PATIENTS = TRAIN_PATIENTS + TEST_PATIENTS

PATIENT_DIRS = {
    patient: os.path.join(BASE_DIR, patient)
    for patient in ALL_PATIENTS
}

for patient, path in PATIENT_DIRS.items():

    if not os.path.exists(path):
        print("WARNING — folder not found:", path)

print("\nDirectory check complete.")


Directory check complete.


In [3]:
# ============================================
# CELL 3 — FIND ALL EDF FILES
# ============================================

patient_edf_files = {}

for patient in ALL_PATIENTS:

    edf_files = sorted(
        glob.glob(
            os.path.join(
                PATIENT_DIRS[patient],
                "*.edf"
            )
        )
    )

    patient_edf_files[patient] = edf_files

    print(
        patient,
        "→",
        len(edf_files),
        "EDF recordings"
    )

total_recordings = sum(
    len(files)
    for files in patient_edf_files.values()
)

print("\nTotal EDF recordings:", total_recordings)

chb01 → 42 EDF recordings
chb02 → 36 EDF recordings
chb03 → 38 EDF recordings
chb04 → 42 EDF recordings
chb05 → 39 EDF recordings
chb06 → 18 EDF recordings
chb07 → 19 EDF recordings
chb08 → 20 EDF recordings
chb09 → 19 EDF recordings
chb10 → 25 EDF recordings
chb11 → 35 EDF recordings
chb12 → 24 EDF recordings
chb13 → 33 EDF recordings
chb14 → 26 EDF recordings
chb15 → 40 EDF recordings
chb16 → 19 EDF recordings
chb17 → 21 EDF recordings
chb18 → 36 EDF recordings
chb19 → 30 EDF recordings
chb20 → 29 EDF recordings
chb21 → 33 EDF recordings
chb22 → 31 EDF recordings
chb23 → 9 EDF recordings
chb24 → 22 EDF recordings

Total EDF recordings: 686


In [4]:
# ============================================
# CELL 4 — SEIZURE SUMMARY PARSER
# ============================================

def read_summary_file(summary_file):

    seizure_info = {}

    with open(summary_file, "r") as f:
        lines = f.readlines()

    current_file = None
    current_seizures = []

    for line in lines:

        line = line.strip()

        if line.startswith("File Name:"):

            if current_file is not None:
                seizure_info[current_file] = current_seizures

            current_file = (
                line.split(":", 1)[1].strip()
            )

            current_seizures = []

        elif line.startswith("Seizure Start Time:"):

            start = float(
                line.split(":", 1)[1]
                .replace("seconds", "")
                .strip()
            )

            current_seizures.append(
                [start, None]
            )

        elif line.startswith("Seizure End Time:"):

            end = float(
                line.split(":", 1)[1]
                .replace("seconds", "")
                .strip()
            )

            if current_seizures:
                current_seizures[-1][1] = end

    if current_file is not None:
        seizure_info[current_file] = current_seizures

    return seizure_info

In [5]:
# ============================================
# CELL 5 — LOAD ALL SUMMARY FILES
# ============================================

summary_info = {}

for patient in ALL_PATIENTS:

    summary_file = os.path.join(
        PATIENT_DIRS[patient],
        f"{patient}-summary.txt"
    )

    if not os.path.exists(summary_file):

        print(
            "WARNING — summary not found:",
            summary_file
        )

        summary_info[patient] = {}

        continue

    summary_info[patient] = (
        read_summary_file(summary_file)
    )

    seizure_count = sum(
        len(v)
        for v in summary_info[patient].values()
    )

    print(
        patient,
        "| recordings:",
        len(summary_info[patient]),
        "| seizures:",
        seizure_count
    )

chb01 | recordings: 42 | seizures: 7
chb02 | recordings: 36 | seizures: 3
chb03 | recordings: 38 | seizures: 7
chb04 | recordings: 42 | seizures: 2
chb05 | recordings: 39 | seizures: 5
chb06 | recordings: 18 | seizures: 0
chb07 | recordings: 19 | seizures: 0
chb08 | recordings: 20 | seizures: 0
chb09 | recordings: 19 | seizures: 0
chb10 | recordings: 25 | seizures: 0
chb11 | recordings: 35 | seizures: 0
chb12 | recordings: 24 | seizures: 0
chb13 | recordings: 33 | seizures: 0
chb14 | recordings: 26 | seizures: 0
chb15 | recordings: 40 | seizures: 0
chb16 | recordings: 19 | seizures: 0
chb17 | recordings: 21 | seizures: 0
chb18 | recordings: 36 | seizures: 0
chb19 | recordings: 30 | seizures: 0
chb20 | recordings: 29 | seizures: 0
chb21 | recordings: 33 | seizures: 0
chb22 | recordings: 31 | seizures: 0
chb23 | recordings: 9 | seizures: 0
chb24 | recordings: 12 | seizures: 16


In [6]:
# ============================================
# CELL 6 — CHECK SAMPLING FREQUENCY
# ============================================

checked = 0

for patient in ALL_PATIENTS:

    files = patient_edf_files[patient]

    if len(files) == 0:
        continue

    edf_file = files[0]

    raw = mne.io.read_raw_edf(
        edf_file,
        preload=False,
        verbose=False
    )

    print(
        patient,
        "|",
        os.path.basename(edf_file),
        "| sfreq:",
        raw.info["sfreq"]
    )

    checked += 1

    if checked >= 5:
        break

chb01 | chb01_01.edf | sfreq: 256.0
chb02 | chb02_01.edf | sfreq: 256.0
chb03 | chb03_01.edf | sfreq: 256.0
chb04 | chb04_01.edf | sfreq: 256.0
chb05 | chb05_01.edf | sfreq: 256.0


C:\Users\PC\AppData\Local\Temp\ipykernel_35456\3340995756.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\PC\AppData\Local\Temp\ipykernel_35456\3340995756.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\PC\AppData\Local\Temp\ipykernel_35456\3340995756.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\PC\AppData\Local\Temp\ipykernel_35456\3340995756.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\PC\AppData\Local\Temp\ipykernel_35456\3340995756.py:16: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. 

In [7]:
# ============================================
# CELL 7 — EPOCH LABELING
# ============================================

EPOCH_DURATION = 5.0
OVERLAP = 1.0
LABEL_THRESHOLD = 0.5


def create_epoch_labels(
    epochs,
    seizure_times,
    threshold=0.5
):

    labels = []
    seizure_ratios = []

    sfreq = epochs.info["sfreq"]

    for event in epochs.events:

        epoch_start = event[0] / sfreq
        epoch_end = (
            epoch_start +
            EPOCH_DURATION
        )

        seizure_duration = 0.0

        for seizure_start, seizure_end in seizure_times:

            overlap_start = max(
                epoch_start,
                seizure_start
            )

            overlap_end = min(
                epoch_end,
                seizure_end
            )

            if overlap_start < overlap_end:

                seizure_duration += (
                    overlap_end -
                    overlap_start
                )

        seizure_ratio = (
            seizure_duration /
            EPOCH_DURATION
        )

        label = int(
            seizure_ratio >= threshold
        )

        labels.append(label)
        seizure_ratios.append(
            seizure_ratio
        )

    return (
        np.array(labels, dtype=np.int64),
        np.array(seizure_ratios)
    )

In [8]:
# ============================================
# CELL 8 — TEST ONE EDF
# ============================================

test_patient = TRAIN_PATIENTS[0]

test_file = patient_edf_files[
    test_patient
][0]

print("Patient:", test_patient)
print("File:", os.path.basename(test_file))

raw = mne.io.read_raw_edf(
    test_file,
    preload=True,
    verbose=False
)

print(
    "Original channels:",
    len(raw.ch_names)
)

print(
    "Sampling frequency:",
    raw.info["sfreq"]
)

# Keep EEG channels
raw.pick("eeg")

print(
    "EEG channels:",
    len(raw.ch_names)
)

# Filter
raw.filter(
    l_freq=1,
    h_freq=30,
    verbose=False
)

# Average reference
raw.set_eeg_reference(
    "average",
    verbose=False
)

# Create 5-second epochs
epochs_test = mne.make_fixed_length_epochs(
    raw,
    duration=EPOCH_DURATION,
    overlap=OVERLAP,
    preload=True,
    verbose=False
)

epochs_test.drop_bad()

print(
    "Epoch shape:",
    epochs_test.get_data().shape
)

C:\Users\PC\AppData\Local\Temp\ipykernel_35456\2599682734.py:14: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


Patient: chb01
File: chb01_01.edf
Original channels: 23
Sampling frequency: 256.0
EEG channels: 23
Epoch shape: (899, 23, 1280)


In [9]:
# ============================================
# CELL 9 — TEST LABELING
# ============================================

test_file_name = os.path.basename(test_file)

seizure_times = summary_info[
    test_patient
].get(
    test_file_name,
    []
)

y_test_file, ratios_test_file = (
    create_epoch_labels(
        epochs_test,
        seizure_times,
        threshold=LABEL_THRESHOLD
    )
)

print("File:", test_file_name)

print(
    "Number of epochs:",
    len(y_test_file)
)

print(
    "Seizure epochs:",
    np.sum(y_test_file == 1)
)

print(
    "Normal epochs:",
    np.sum(y_test_file == 0)
)

File: chb01_01.edf
Number of epochs: 899
Seizure epochs: 0
Normal epochs: 899


In [10]:
# ============================================
# CELL 10 — HDF5
# ============================================

import h5py

print("h5py version:", h5py.__version__)

h5py version: 3.14.0


In [11]:
# ============================================
# CELL 11 — CREATE HDF5 DATASET
# ============================================

H5_FILE = os.path.join(
    BASE_DIR,
    "chb01_chb24_raw_eeg.h5"
)

print("Dataset file:")
print(H5_FILE)

Dataset file:
D:\EEG MIT\chb-mit-scalp-eeg-database-1.0.0\chb-mit-scalp-eeg-database-1.0.0\chb01_chb24_raw_eeg.h5


In [ ]:
# ============================================
# CELL 12 — PROCESS ALL RAW EEG
# ============================================

import h5py

if os.path.exists(H5_FILE):
    os.remove(H5_FILE)

with h5py.File(H5_FILE, "w") as h5:

    X_dataset = None
    y_dataset = None
    patient_dataset = None

    total_epochs = 0

    for patient in ALL_PATIENTS:

        print("\n================================")
        print("PROCESSING", patient)
        print("================================")

        files = patient_edf_files[patient]

        for file_index, edf_file in enumerate(files):

            file_name = os.path.basename(edf_file)

            print(
                f"[{file_index + 1}/{len(files)}]",
                file_name
            )

            try:

                # --------------------------------
                # Load EEG
                # --------------------------------

                raw = mne.io.read_raw_edf(
                    edf_file,
                    preload=True,
                    verbose=False
                )

                # --------------------------------
                # Keep EEG only
                # --------------------------------

                #raw.pick("eeg") for all 23 channel 

                # --------------------------------
                # Select 18 EEG channels
                # --------------------------------

                CHANNELS_18 = [
                    "FP1-F7",
                    "F7-T7",
                    "T7-P7",
                    "P7-O1",
                    "FP1-F3",
                    "F3-C3",
                    "C3-P3",
                    "P3-O1",
                    "FP2-F4",
                    "F4-C4",
                    "C4-P4",
                    "P4-O2",
                    "FP2-F8",
                    "F8-T8",
                    "T8-P8",
                    "P8-O2",
                    "FZ-CZ",
                    "CZ-PZ"
                ]

                # Check that all channels exist
                missing = [
                    ch for ch in CHANNELS_18
                    if ch not in raw.ch_names
                ]

                if missing:
                    print("Missing channels:", missing)
                    continue

                raw.pick(CHANNELS_18)

                # --------------------------------
                # Check channel count
                # --------------------------------

                if len(raw.ch_names) != 18:

                    print(
                        "  WARNING:",
                        len(raw.ch_names),
                        "EEG channels selected"
                    )

                # --------------------------------
                # Filter
                # --------------------------------

                raw.filter(
                    l_freq=1,
                    h_freq=30,
                    verbose=False
                )

                # --------------------------------
                # Average reference
                # --------------------------------

                raw.set_eeg_reference(
                    "average",
                    verbose=False
                )

                # --------------------------------
                # Create epochs
                # --------------------------------

                epochs = (
                    mne.make_fixed_length_epochs(
                        raw,
                        duration=EPOCH_DURATION,
                        overlap=OVERLAP,
                        preload=True,
                        verbose=False
                    )
                )

                epochs.drop_bad()

                X_file = epochs.get_data()

                # --------------------------------
                # Labels
                # --------------------------------

                seizure_times = summary_info[
                    patient
                ].get(
                    file_name,
                    []
                )

                y_file, seizure_ratios = (
                    create_epoch_labels(
                        epochs,
                        seizure_times,
                        threshold=LABEL_THRESHOLD
                    )
                )

                # --------------------------------
                # Check shapes
                # --------------------------------

                if X_file.shape[0] != len(y_file):

                    print(
                        "  ERROR: X/y mismatch"
                    )

                    continue

                # --------------------------------
                # Create HDF5 datasets
                # --------------------------------

                if X_dataset is None:

                    X_dataset = h5.create_dataset(
                        "X",
                        shape=(0,) + X_file.shape[1:],
                        maxshape=(
                            None,
                        ) + X_file.shape[1:],
                        dtype="float32",
                        chunks=True,
                        compression="gzip"
                    )

                    y_dataset = h5.create_dataset(
                        "y",
                        shape=(0,),
                        maxshape=(None,),
                        dtype="int8",
                        chunks=True,
                        compression="gzip"
                    )

                    patient_dataset = h5.create_dataset(
                        "patient",
                        shape=(0,),
                        maxshape=(None,),
                        dtype="S5",
                        chunks=True,
                        compression="gzip"
                    )

                # --------------------------------
                # Append X
                # --------------------------------

                old_size = X_dataset.shape[0]
                new_size = (
                    old_size +
                    X_file.shape[0]
                )

                X_dataset.resize(
                    new_size,
                    axis=0
                )

                X_dataset[
                    old_size:new_size
                ] = X_file.astype(
                    np.float32
                )

                # --------------------------------
                # Append labels
                # --------------------------------

                y_dataset.resize(
                    new_size,
                    axis=0
                )

                y_dataset[
                    old_size:new_size
                ] = y_file.astype(
                    np.int8
                )

                # --------------------------------
                # Append patient IDs
                # --------------------------------

                patient_dataset.resize(
                    new_size,
                    axis=0
                )

                patient_dataset[
                    old_size:new_size
                ] = np.array(
                    [patient.encode()] *
                    len(y_file)
                )

                total_epochs = new_size

                print(
                    "  Epochs:",
                    len(y_file),
                    "| Seizure:",
                    np.sum(y_file == 1),
                    "| Normal:",
                    np.sum(y_file == 0)
                )

                # Free memory
                del raw
                del epochs
                del X_file

            except Exception as e:

                print(
                    "  ERROR:",
                    file_name,
                    "→",
                    str(e)
                )

    print("\n================================")
    print("DATASET CREATION COMPLETE")
    print("================================")

    print(
        "Total epochs:",
        total_epochs
    )


In [13]:
# ============================================
# CELL 13 — VERIFY DATASET
# ============================================

with h5py.File(H5_FILE, "r") as h5:

    print("X shape:", h5["X"].shape)
    print("y shape:", h5["y"].shape)
    print(
        "patient shape:",
        h5["patient"].shape
    )

    y = h5["y"][:]

    print(
        "\nTotal seizure epochs:",
        np.sum(y == 1)
    )

    print(
        "Total normal epochs:",
        np.sum(y == 0)
    )

    print(
        "Seizure percentage:",
        np.mean(y == 1) * 100
    )

X shape: (25176, 18, 1280)
y shape: (25176,)
patient shape: (25176,)

Total seizure epochs: 0
Total normal epochs: 25176
Seizure percentage: 0.0


In [14]:
# ============================================
# CELL 14 — PATIENT DISTRIBUTION
# ============================================

with h5py.File(H5_FILE, "r") as h5:

    patients = h5["patient"][:]
    y = h5["y"][:]

    for patient in ALL_PATIENTS:

        patient_bytes = patient.encode()

        mask = (
            patients == patient_bytes
        )

        print(
            patient,
            "| epochs:",
            np.sum(mask),
            "| seizure:",
            np.sum(y[mask] == 1),
            "| normal:",
            np.sum(y[mask] == 0)
        )

chb01 | epochs: 0 | seizure: 0 | normal: 0
chb02 | epochs: 0 | seizure: 0 | normal: 0
chb03 | epochs: 0 | seizure: 0 | normal: 0
chb04 | epochs: 0 | seizure: 0 | normal: 0
chb05 | epochs: 0 | seizure: 0 | normal: 0
chb06 | epochs: 0 | seizure: 0 | normal: 0
chb07 | epochs: 0 | seizure: 0 | normal: 0
chb08 | epochs: 0 | seizure: 0 | normal: 0
chb09 | epochs: 0 | seizure: 0 | normal: 0
chb10 | epochs: 0 | seizure: 0 | normal: 0
chb11 | epochs: 0 | seizure: 0 | normal: 0
chb12 | epochs: 0 | seizure: 0 | normal: 0
chb13 | epochs: 19778 | seizure: 0 | normal: 19778
chb14 | epochs: 0 | seizure: 0 | normal: 0
chb15 | epochs: 900 | seizure: 0 | normal: 900
chb16 | epochs: 1798 | seizure: 0 | normal: 1798
chb17 | epochs: 899 | seizure: 0 | normal: 899
chb18 | epochs: 902 | seizure: 0 | normal: 902
chb19 | epochs: 899 | seizure: 0 | normal: 899
chb20 | epochs: 0 | seizure: 0 | normal: 0
chb21 | epochs: 0 | seizure: 0 | normal: 0
chb22 | epochs: 0 | seizure: 0 | normal: 0
chb23 | epochs: 0 | seiz

In [15]:
# ============================================
# CELL 15 — PATIENT-INDEPENDENT SPLIT
# ============================================

with h5py.File(H5_FILE, "r") as h5:

    patients = h5["patient"][:]

    train_mask = np.isin(
        patients,
        np.array(
            [p.encode() for p in TRAIN_PATIENTS]
        )
    )

    test_mask = np.isin(
        patients,
        np.array(
            [p.encode() for p in TEST_PATIENTS]
        )
    )

    train_indices = np.where(
        train_mask
    )[0]

    test_indices = np.where(
        test_mask
    )[0]

print("Training epochs:", len(train_indices))
print("Testing epochs:", len(test_indices))

print(
    "\nTraining patients:",
    TRAIN_PATIENTS
)

print(
    "Testing patients:",
    TEST_PATIENTS
)

Training epochs: 23375
Testing epochs: 1801

Training patients: ['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14', 'chb15', 'chb16', 'chb17']
Testing patients: ['chb18', 'chb19', 'chb20', 'chb21', 'chb22', 'chb23', 'chb24']


In [16]:
# ============================================
# CELL 16 — CHECK PATIENT LEAKAGE
# ============================================

with h5py.File(H5_FILE, "r") as h5:

    patients = h5["patient"][:]

    train_patients_found = set(
        p.decode()
        for p in np.unique(
            patients[train_indices]
        )
    )

    test_patients_found = set(
        p.decode()
        for p in np.unique(
            patients[test_indices]
        )
    )

print(
    "Train patients:",
    sorted(train_patients_found)
)

print(
    "Test patients:",
    sorted(test_patients_found)
)

print(
    "\nPatient overlap:",
    train_patients_found.intersection(
        test_patients_found
    )
)

Train patients: ['chb13', 'chb15', 'chb16', 'chb17']
Test patients: ['chb18', 'chb19']

Patient overlap: set()


In [17]:
# ============================================================
# CELL 17 — Calculate training normalization statistics
# ============================================================

import numpy as np
import h5py

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SAMPLE_SIZE = 10000

# ------------------------------------------------------------
# Open HDF5 dataset
# ------------------------------------------------------------

with h5py.File(H5_FILE, "r") as h5:

    X = h5["X"]
    y = h5["y"]
    patients = h5["patient"]

    print("Full X shape:", X.shape)
    print("Full y shape:", y.shape)

    # --------------------------------------------------------
    # Check that dataset contains 18 channels
    # --------------------------------------------------------

    if X.shape[1] != 18:
        raise ValueError(
            f"Expected 18 EEG channels, but HDF5 contains "
            f"{X.shape[1]} channels.\n"
            f"You need to rebuild the HDF5 dataset using "
            f"CHANNELS_18 before continuing."
        )

    if X.shape[2] != 1280:
        raise ValueError(
            f"Expected 1280 samples per epoch, but found "
            f"{X.shape[2]}."
        )

    # --------------------------------------------------------
    # Find training indices
    # CHB01–CHB17 = training
    # CHB18–CHB24 = test
    # --------------------------------------------------------

    patient_array = np.array([
        p.decode() if isinstance(p, bytes) else str(p)
        for p in patients[:]
    ])

    train_mask = np.isin(
        patient_array,
        TRAIN_PATIENTS
    )

    train_indices = np.where(train_mask)[0]

    print("Training epochs:", len(train_indices))

    # --------------------------------------------------------
    # Randomly select training epochs
    # --------------------------------------------------------

    sample_size = min(
        SAMPLE_SIZE,
        len(train_indices)
    )

    sample_indices = np.random.choice(
        train_indices,
        size=sample_size,
        replace=False
    )

    # IMPORTANT:
    # h5py requires indices to be in increasing order
    sample_indices = np.sort(sample_indices)

    print("Normalization sample size:", len(sample_indices))

    # --------------------------------------------------------
    # Load sampled training epochs
    # --------------------------------------------------------

    X_sample = X[sample_indices]

    print("Sample shape:", X_sample.shape)

# ============================================================
# Calculate per-channel mean and standard deviation
# ============================================================

# X_sample shape:
#
#     (N, 18, 1280)
#
# axis=0 → across epochs
# axis=2 → across time samples
#
# Result:
#
#     train_mean → (18,)
#     train_std  → (18,)

train_mean = X_sample.mean(
    axis=(0, 2)
)

train_std = X_sample.std(
    axis=(0, 2)
)

# ------------------------------------------------------------
# Prevent division by zero
# ------------------------------------------------------------

train_std[train_std < 1e-8] = 1.0

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nNormalization statistics calculated.")

print("\nMean shape:")
print(train_mean.shape)

print("\nStd shape:")
print(train_std.shape)

print("\nFirst 5 channel means:")
print(train_mean[:5])

print("\nFirst 5 channel stds:")
print(train_std[:5])

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n========== CHECK ==========")
print("Dataset channels :", X_sample.shape[1])
print("Samples/epoch    :", X_sample.shape[2])
print("Mean shape       :", train_mean.shape)
print("Std shape        :", train_std.shape)

Full X shape: (25176, 18, 1280)
Full y shape: (25176,)
Training epochs: 23375
Normalization sample size: 10000
Sample shape: (10000, 18, 1280)

Normalization statistics calculated.

Mean shape:
(18,)

Std shape:
(18,)

First 5 channel means:
[-1.2777009e-08  1.3047625e-08 -7.0375199e-09 -3.3996652e-09
 -2.5878776e-08]

First 5 channel stds:
[4.9101171e-05 4.5289667e-05 4.8440641e-05 4.6657799e-05 5.6412518e-05]

========== CHECK ==========
Dataset channels : 18
Samples/epoch    : 1280
Mean shape       : (18,)
Std shape        : (18,)


In [18]:
# ============================================
# CELL 18 — SNN LIBRARY
# ============================================

import torch

print("PyTorch version:", torch.__version__)

try:
    import snntorch as snn

    print(
        "snnTorch version:",
        snn.__version__
    )

except ImportError:

    print(
        "snnTorch is not installed."
    )

    print(
        "Install it with:"
    )

    print(
        "!pip install snntorch"
    )

PyTorch version: 2.14.0+cpu
snnTorch version: 1.0.0


In [22]:
!pip install numpy scipy pandas matplotlib scikit-learn h5py mne


In [23]:
!uv add torch torchvision torchaudio


Resolved 64 packages in 24ms
Audited 44 packages in 13ms


In [24]:
!uv --version

uv 0.10.2 (a788db7e5 2026-02-10)


In [25]:
!uv add snntorch

Resolved 64 packages in 2ms
Audited 44 packages in 1ms


In [19]:
import torch
import snntorch

print("PyTorch:", torch.__version__)
print("snnTorch:", snntorch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
snnTorch: 1.0.0
Device: cpu


In [20]:
# ============================================================
# CELL 18 — PyTorch Dataset for CHB-MIT Raw EEG
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np


class CHBMITDataset(Dataset):

    def __init__(
        self,
        h5_file,
        indices,
        mean,
        std
    ):

        self.h5_file = h5_file

        # Store indices
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )

        # Store normalization statistics
        self.mean = np.asarray(
            mean,
            dtype=np.float32
        )

        self.std = np.asarray(
            std,
            dtype=np.float32
        )

        # HDF5 file will be opened lazily
        self.h5 = None

    def _open_h5(self):

        if self.h5 is None:
            self.h5 = h5py.File(
                self.h5_file,
                "r"
            )

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, idx):

        self._open_h5()

        real_idx = self.indices[idx]

        # Read one EEG epoch
        x = self.h5["X"][real_idx]

        # Read label
        y = self.h5["y"][real_idx]

        # Convert to float32
        x = x.astype(
            np.float32
        )

        # ----------------------------------------------------
        # Channel-wise normalization
        #
        # x shape:
        #     (18, 1280)
        #
        # mean shape:
        #     (18,)
        #
        # reshape mean/std to:
        #     (18, 1)
        # ----------------------------------------------------

        x = (
            x - self.mean[:, None]
        ) / self.std[:, None]

        # Convert to PyTorch tensors
        x = torch.from_numpy(x)

        y = torch.tensor(
            y,
            dtype=torch.float32
        )

        return x, y


print("CHB-MIT PyTorch Dataset class created.")

CHB-MIT PyTorch Dataset class created.


In [21]:
# ============================================================
# CELL 19 — Create train/test datasets
# ============================================================

train_dataset = CHBMITDataset(
    h5_file=H5_FILE,
    indices=train_indices,
    mean=train_mean,
    std=train_std
)

test_dataset = CHBMITDataset(
    h5_file=H5_FILE,
    indices=test_indices,
    mean=train_mean,
    std=train_std
)

print("Training samples:", len(train_dataset))
print("Testing samples :", len(test_dataset))

Training samples: 23375
Testing samples : 1801


In [22]:
# ============================================================
# CELL 20 — Test Dataset
# ============================================================

x, y = train_dataset[0]

print("EEG shape:", x.shape)
print("Label:", y)
print("Data type:", x.dtype)

print(
    "Mean approximately:",
    x.mean().item()
)

print(
    "Std approximately:",
    x.std().item()
)

EEG shape: torch.Size([18, 1280])
Label: tensor(0.)
Data type: torch.float32
Mean approximately: -0.004768828395754099
Std approximately: 1.1214008331298828


In [23]:
# ============================================================
# CELL 21 — PyTorch DataLoaders
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))

Train batches: 731
Test batches : 57


In [31]:
# ============================================================
# CELL 22 — Verify a training batch
# ============================================================

# x_batch, y_batch = next(
#     iter(train_loader)
# )

# print("X batch shape:", x_batch.shape)
# print("Y batch shape:", y_batch.shape)

# print("X dtype:", x_batch.dtype)
# print("Y dtype:", y_batch.dtype)

# print("Number of seizure samples in batch:",
#       int(y_batch.sum().item()))

# print("Number of normal samples in batch:",
#       int((y_batch == 0).sum().item()))

In [24]:
# ============================================================
# CELL 23 — 1D CNN Feature Extractor
# ============================================================

import torch
import torch.nn as nn


class EEG_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            # ------------------------------------------------
            # Block 1
            # ------------------------------------------------

            nn.Conv1d(
                in_channels=18,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),

            nn.BatchNorm1d(32),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            # ------------------------------------------------
            # Block 2
            # ------------------------------------------------

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            # ------------------------------------------------
            # Block 3
            # ------------------------------------------------

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.MaxPool1d(
                kernel_size=2
            )
        )

    def forward(self, x):

        x = self.features(x)

        return x


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

cnn = EEG_CNN()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

cnn = cnn.to(device)

print("Device:", device)
print(cnn)

Device: cpu
EEG_CNN(
  (features): Sequential(
    (0): Conv1d(18, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
)


In [ ]:
# ============================================================
CELL 24 — Test CNN with one batch
============================================================

x_batch, y_batch = next(
    iter(train_loader)
)

x_batch = x_batch.to(device)

with torch.no_grad():

    cnn_output = cnn(x_batch)

print("Input shape:")
print(x_batch.shape)

print("\nCNN output shape:")
print(cnn_output.shape)

In [ ]:
# ============================================================
# CELL 24 — Test CNN with One Small Batch
# ============================================================

import torch
from torch.utils.data import DataLoader

print("=" * 60)
print("CELL 24 — CNN TEST")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check device
# ------------------------------------------------------------

print("\n[1] Checking device...")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

# ------------------------------------------------------------
# 2. Move CNN to device
# ------------------------------------------------------------

print("\n[2] Moving CNN to device...")

cnn = cnn.to(device)
cnn.eval()

print("CNN is on:", device)

# ------------------------------------------------------------
# 3. Get one batch
# ------------------------------------------------------------

print("\n[3] Loading one batch...")

try:
    x_batch, y_batch = next(iter(train_loader))

    print("Batch loaded successfully!")

except Exception as e:
    print("ERROR while loading batch:")
    print(e)
    raise

# ------------------------------------------------------------
# 4. Display original batch information
# ------------------------------------------------------------

print("\n[4] Original batch information")

print("Input shape :", x_batch.shape)
print("Label shape :", y_batch.shape)
print("Input dtype :", x_batch.dtype)
print("Label dtype :", y_batch.dtype)

# ------------------------------------------------------------
# 5. Use only 1 or 2 images for testing
# ------------------------------------------------------------

TEST_BATCH_SIZE = 2

x_test = x_batch[:TEST_BATCH_SIZE]
y_test = y_batch[:TEST_BATCH_SIZE]

print("\n[5] Using small test batch")

print("Test input shape :", x_test.shape)
print("Test label shape :", y_test.shape)

# ------------------------------------------------------------
# 6. Move test batch to device
# ------------------------------------------------------------

print("\n[6] Moving test batch to", device)

x_test = x_test.to(device)

print("Batch successfully moved to device.")

# ------------------------------------------------------------
# 7. Run CNN
# ------------------------------------------------------------

print("\n[7] Running CNN...")
print("Please wait...")

try:

    with torch.no_grad():

        cnn_output = cnn(x_test)

    print("CNN forward pass completed!")

except RuntimeError as e:

    print("\nCNN ERROR:")
    print(e)
    raise

# ------------------------------------------------------------
# 8. Display output
# ------------------------------------------------------------

print("\n[8] CNN results")

print("-" * 60)

print("Input shape :")
print(x_test.shape)

print("\nOutput shape :")
print(cnn_output.shape)

print("\nOutput:")
print(cnn_output)

# ------------------------------------------------------------
# 9. Check for NaN / Inf
# ------------------------------------------------------------

print("\n[9] Checking output...")

if torch.isnan(cnn_output).any():
    print("WARNING: CNN output contains NaN values!")
else:
    print("No NaN values.")

if torch.isinf(cnn_output).any():
    print("WARNING: CNN output contains Inf values!")
else:
    print("No Inf values.")

# ------------------------------------------------------------
# 10. Final message
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CNN TEST COMPLETED SUCCESSFULLY")
print("=" * 60)


CELL 24 — CNN TEST

[1] Checking device...
CUDA not available. Using CPU.

[2] Moving CNN to device...
CNN is on: cpu

[3] Loading one batch...


In [25]:
# ============================================================
# CELL 25 — Convert CNN features to SNN time format
# ============================================================

cnn_features = cnn_output.permute(
    2, 0, 1
)

print("CNN features:", cnn_output.shape)
print("SNN input format:", cnn_features.shape)

NameError: name 'cnn_output' is not defined

In [ ]:
# ============================================================
# CELL 26 — Test LIF neuron
# ============================================================

import snntorch as snn

lif = snn.Leaky(
    beta=0.9
).to(device)

print(lif)

In [ ]:
# ============================================================
# CELL 27 — Run CNN features through LIF
# ============================================================

mem = lif.init_leaky()

spike_record = []

for step in range(
    cnn_features.shape[0]
):

    current = cnn_features[step]

    spike, mem = lif(
        current,
        mem
    )

    spike_record.append(
        spike
    )

spike_record = torch.stack(
    spike_record
)

print("Spike output shape:")
print(spike_record.shape)

In [ ]:
# ============================================================
# CELL 28 — Complete CNN → LIF SNN Model
# ============================================================

import torch
import torch.nn as nn
import snntorch as snn


class CNN_LIF_SNN(nn.Module):

    def __init__(self, beta=0.9):

        super().__init__()

        # ====================================================
        # CNN FEATURE EXTRACTOR
        # ====================================================

        self.cnn = nn.Sequential(

            # Block 1
            nn.Conv1d(
                in_channels=18,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 2
            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 3
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # ====================================================
        # LIF SNN
        # ====================================================

        self.lif = snn.Leaky(
            beta=beta
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        # ----------------------------------------------------
        # x:
        # [batch, 18, 1280]
        # ----------------------------------------------------

        x = self.cnn(x)

        # ----------------------------------------------------
        # CNN output:
        # [batch, 128, 160]
        # ----------------------------------------------------

        # Convert to:
        # [time, batch, features]

        x = x.permute(
            2, 0, 1
        )

        # ----------------------------------------------------
        # LIF
        # ----------------------------------------------------

        mem = self.lif.init_leaky()

        spike_record = []

        for t in range(
            x.size(0)
        ):

            spike, mem = self.lif(
                x[t],
                mem
            )

            spike_record.append(
                spike
            )

        # [time, batch, features]

        spikes = torch.stack(
            spike_record
        )

        # ----------------------------------------------------
        # Temporal spike aggregation
        # ----------------------------------------------------

        spike_count = spikes.sum(
            dim=0
        )

        # Average firing rate
        spike_rate = spike_count / spikes.size(0)

        # ----------------------------------------------------
        # Final classifier
        # ----------------------------------------------------

        logits = self.classifier(
            spike_rate
        )

        return logits

In [ ]:
# ============================================================
# CELL 29 — Create Model
# ============================================================

model = CNN_LIF_SNN(
    beta=0.9
).to(device)

print(model)

In [ ]:
# ============================================================
# CELL 30 — Test Complete Model
# ============================================================

x_batch, y_batch = next(
    iter(train_loader)
)

x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

model.eval()

with torch.no_grad():

    logits = model(
        x_batch
    )

print("Input:")
print(x_batch.shape)

print("\nOutput:")
print(logits.shape)

print("\nLogits:")
print(logits[:10])

In [ ]:
# ============================================================
# CELL 31 — Seizure Probability
# ============================================================

probabilities = torch.sigmoid(
    logits
)

print(
    "Seizure probabilities:"
)

print(
    probabilities[:10].squeeze()
)

In [ ]:
# ============================================================
# CELL 32 — Calculate Training Class Weights
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import h5py

# ------------------------------------------------------------
# Read training labels only
# ------------------------------------------------------------

with h5py.File(H5_FILE, "r") as h5:

    y_all = h5["y"][:]

# ------------------------------------------------------------
# Select CHB01–CHB17 only
# ------------------------------------------------------------

patient_array = np.array([
    p.decode() if isinstance(p, bytes) else str(p)
    for p in h5["patient"][:]
]) if False else None

# We already have train_indices, so use them directly
y_train = y_all[train_indices]

# ------------------------------------------------------------
# Count classes
# ------------------------------------------------------------

normal_count = np.sum(y_train == 0)
seizure_count = np.sum(y_train == 1)

print("Training normal epochs :", normal_count)
print("Training seizure epochs:", seizure_count)

print(
    "\nSeizure percentage:",
    seizure_count / len(y_train) * 100,
    "%"
)

# ------------------------------------------------------------
# Calculate positive-class weight
# ------------------------------------------------------------

pos_weight_value = (
    normal_count / seizure_count
)

print(
    "\nPositive class weight:",
    pos_weight_value
)

# ------------------------------------------------------------
# Convert to PyTorch tensor
# ------------------------------------------------------------

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

print(
    "PyTorch pos_weight:",
    pos_weight
)

In [ ]:
# ============================================================
# CELL 33 — Weighted Loss Function
# ============================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print(criterion)

In [ ]:
# ============================================================
# CELL 34 — Optimizer
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

print(optimizer)

In [ ]:
# ============================================================
# CELL 35 — Model Parameters
# ============================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

In [ ]:
# ============================================================
# CELL 36 — Patient-Level Train / Validation Split
# ============================================================

import h5py
import numpy as np

# ------------------------------------------------------------
# Development patients
# ------------------------------------------------------------

TRAIN_PATIENTS_DEV = [
    f"chb{i:02d}"
    for i in range(1, 16)
]

VAL_PATIENTS = [
    "chb16",
    "chb17"
]

TEST_PATIENTS = [
    f"chb{i:02d}"
    for i in range(18, 25)
]

print("Development training patients:")
print(TRAIN_PATIENTS_DEV)

print("\nValidation patients:")
print(VAL_PATIENTS)

print("\nFinal test patients:")
print(TEST_PATIENTS)


# ------------------------------------------------------------
# Read patient information from HDF5
# ------------------------------------------------------------

with h5py.File(H5_FILE, "r") as h5:

    patient_array = np.array([
        p.decode() if isinstance(p, bytes) else str(p)
        for p in h5["patient"][:]
    ])


# ------------------------------------------------------------
# Create masks
# ------------------------------------------------------------

train_mask = np.isin(
    patient_array,
    TRAIN_PATIENTS_DEV
)

val_mask = np.isin(
    patient_array,
    VAL_PATIENTS
)

test_mask = np.isin(
    patient_array,
    TEST_PATIENTS
)


# ------------------------------------------------------------
# Convert masks to indices
# ------------------------------------------------------------

train_indices = np.where(
    train_mask
)[0]

val_indices = np.where(
    val_mask
)[0]

test_indices = np.where(
    test_mask
)[0]


# ------------------------------------------------------------
# Print sizes
# ------------------------------------------------------------

print("\n========== DATA SPLIT ==========")

print(
    "Training epochs  :",
    len(train_indices)
)

print(
    "Validation epochs:",
    len(val_indices)
)

print(
    "Test epochs      :",
    len(test_indices)
)

In [ ]:
# ============================================================
# CELL 37 — Verify Patient Independence
# ============================================================

train_patients_found = set(
    patient_array[train_indices]
)

val_patients_found = set(
    patient_array[val_indices]
)

test_patients_found = set(
    patient_array[test_indices]
)


print("Training patients:")
print(sorted(train_patients_found))

print("\nValidation patients:")
print(sorted(val_patients_found))

print("\nTest patients:")
print(sorted(test_patients_found))


# ------------------------------------------------------------
# Check overlap
# ------------------------------------------------------------

train_val_overlap = (
    train_patients_found &
    val_patients_found
)

train_test_overlap = (
    train_patients_found &
    test_patients_found
)

val_test_overlap = (
    val_patients_found &
    test_patients_found
)


print("\n========== LEAKAGE CHECK ==========")

print(
    "Train ∩ Validation:",
    train_val_overlap
)

print(
    "Train ∩ Test:",
    train_test_overlap
)

print(
    "Validation ∩ Test:",
    val_test_overlap
)

In [ ]:
# ============================================================
# CELL 38 — Class Distribution
# ============================================================

with h5py.File(H5_FILE, "r") as h5:

    y_all = h5["y"][:]


def print_class_distribution(
    name,
    indices
):

    labels = y_all[indices]

    normal = np.sum(labels == 0)
    seizure = np.sum(labels == 1)

    total = len(labels)

    print(f"\n{name}")
    print("-" * 35)

    print("Total   :", total)
    print("Normal  :", normal)
    print("Seizure :", seizure)

    if total > 0:
        print(
            "Seizure %:",
            f"{seizure / total * 100:.4f}%"
        )


print_class_distribution(
    "TRAIN",
    train_indices
)

print_class_distribution(
    "VALIDATION",
    val_indices
)

print_class_distribution(
    "TEST",
    test_indices
)

In [ ]:
#Because we changed the training set from CHB01–17 to CHB01–15 for validation, you should rerun Cell 32 after Cell 38.

In [ ]:
# ============================================================
# CELL 32 — Calculate Training Class Weights
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import h5py

# ------------------------------------------------------------
# Read training labels only
# ------------------------------------------------------------

with h5py.File(H5_FILE, "r") as h5:

    y_all = h5["y"][:]

# ------------------------------------------------------------
# Select CHB01–CHB17 only
# ------------------------------------------------------------

patient_array = np.array([
    p.decode() if isinstance(p, bytes) else str(p)
    for p in h5["patient"][:]
]) if False else None

# We already have train_indices, so use them directly
y_train = y_all[train_indices]

# ------------------------------------------------------------
# Count classes
# ------------------------------------------------------------

normal_count = np.sum(y_train == 0)
seizure_count = np.sum(y_train == 1)

print("Training normal epochs :", normal_count)
print("Training seizure epochs:", seizure_count)

print(
    "\nSeizure percentage:",
    seizure_count / len(y_train) * 100,
    "%"
)

# ------------------------------------------------------------
# Calculate positive-class weight
# ------------------------------------------------------------

pos_weight_value = (
    normal_count / seizure_count
)

print(
    "\nPositive class weight:",
    pos_weight_value
)

# ------------------------------------------------------------
# Convert to PyTorch tensor
# ------------------------------------------------------------

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

print(
    "PyTorch pos_weight:",
    pos_weight
)

In [ ]:
# ============================================================
# CELL 39 — Train / Validation / Test Datasets
# ============================================================

train_dataset = CHBMITDataset(
    h5_file=H5_FILE,
    indices=train_indices,
    mean=train_mean,
    std=train_std
)

val_dataset = CHBMITDataset(
    h5_file=H5_FILE,
    indices=val_indices,
    mean=train_mean,
    std=train_std
)

test_dataset = CHBMITDataset(
    h5_file=H5_FILE,
    indices=test_indices,
    mean=train_mean,
    std=train_std
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================================
# CELL 40 — Train / Validation / Test DataLoaders
# ============================================================

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# CELL 41 — Final Training Setup Check
# ============================================================

print("Device:", device)

print("\nModel:")
print(model)

print("\nPositive class weight:")
print(pos_weight)

print("\nLearning rate:")
print(optimizer.param_groups[0]["lr"])

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# CELL 42 — CNN → LIF SNN Training Loop
# ============================================================

import torch
import numpy as np
import copy
import time

# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

NUM_EPOCHS = 20

best_val_loss = float("inf")
best_model_state = None

train_losses = []
val_losses = []

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for x_batch, y_batch in train_loader:

        # Move data to GPU/CPU
        x_batch = x_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        # Make label shape [batch, 1]
        y_batch = y_batch.unsqueeze(1)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(x_batch)

        # Calculate loss
        loss = criterion(
            logits,
            y_batch
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate loss
        batch_size = x_batch.size(0)

        running_train_loss += (
            loss.item() * batch_size
        )

        train_samples += batch_size

    epoch_train_loss = (
        running_train_loss /
        train_samples
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    with torch.no_grad():

        for x_batch, y_batch in val_loader:

            x_batch = x_batch.to(
                device,
                non_blocking=True
            )

            y_batch = y_batch.to(
                device,
                non_blocking=True
            )

            y_batch = y_batch.unsqueeze(1)

            # Forward pass
            logits = model(x_batch)

            # Validation loss
            loss = criterion(
                logits,
                y_batch
            )

            batch_size = x_batch.size(0)

            running_val_loss += (
                loss.item() * batch_size
            )

            val_samples += batch_size

    epoch_val_loss = (
        running_val_loss /
        val_samples
    )

    # --------------------------------------------------------
    # Save losses
    # --------------------------------------------------------

    train_losses.append(
        epoch_train_loss
    )

    val_losses.append(
        epoch_val_loss
    )

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print(
            f"  ✓ Best model saved "
            f"(val loss = {best_val_loss:.6f})"
        )

    # --------------------------------------------------------
    # Epoch time
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {epoch_train_loss:.6f} "
        f"| Val Loss: {epoch_val_loss:.6f} "
        f"| Time: {elapsed:.1f}s"
    )


# ============================================================
# Restore best model
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

print("\nTraining finished.")
print(
    f"Best validation loss: "
    f"{best_val_loss:.6f}"
)

In [ ]:
# ============================================================
# CELL 43 — Training Curves
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN-LIF SNN Training")

plt.legend()
plt.grid(True)

plt.show()